In [29]:
import pandas as pd, numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

m = pd.read_csv("long.csv")
reg = m.dropna(subset=["debt_lag", "pb"]).copy()

def era(y):
    if 1919 <= y <= 1939: return "Interwar (1919–39)"
    if 1946 <= y <= 1973: return "Post-war (1946–73)"
    if 1974 <= y <= 1999: return "1974–99"
    if 2000 <= y <= 2016: return "2000–16"
    return None
reg["era"] = reg["year"].apply(era)
reg = reg.dropna(subset=["era"])
order = ["Interwar (1919–39)", "Post-war (1946–73)", "1974–99", "2000–16"]

base = alt.Chart(reg).encode(
    x=alt.X("debt_lag:Q", title="Debt (% GDP), prev. year"),
    y=alt.Y("pb:Q", title="Primary balance (% GDP)"))

pts = base.mark_point(filled=True, size=40, opacity=0.55,
    color="#179FDB").encode()

fit = base.transform_regression(
    "debt_lag", "pb").mark_line(strokeWidth=2.5, color="#E6224B")

zero = alt.Chart(reg).mark_rule(color="#c9c9c9", strokeDash=[2,3]).encode(
    y=alt.datum(0))

panel = (zero + pts + fit).properties(width=210, height=190)

chart = panel.facet(
    facet=alt.Facet("era:N", sort=order, title=None,
                    header=alt.Header(labelFontSize=13, labelFontWeight="bold",
                                      labelColor="#122b39")),
    columns=2,
    title=alt.Title(
        text="Source: author's calculation; Bank of England, A Millennium of Macroeconomic Data",
        subtitle=[
            "UK primary balance vs previous-year debt, by era. Red line = fitted response.",
            "Upward slope = debt-stabilising. The slope flattens after the 1970s and turns negative since 2000.",
        ],
        orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
        color="#676A86", subtitleColor="#676A86", dy=12),
).configure(background="white", font="Circular Std").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path="Charts", name="fiscal_reaction_scatter", svg=True)
chart.save("Charts/fiscal_reaction_scatter.png", scale_factor=2.0)
chart

alt.FacetChart(...)